# FinBERT news scoring — Indian Market Nexus
Self-contained. **No repo needed.** Upload your `announcements.parquet`, run all cells, download `news_signal.parquet`, drop it into your local `data/processed/`.

**First: set the GPU.**  Runtime menu → *Change runtime type* → Hardware accelerator → **T4 GPU** → Save.

In [ ]:
# 1. Install + confirm GPU
!pip install -q transformers
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (go set the GPU runtime!)")

## 2. Upload `announcements.parquet`
Click *Choose Files* and pick the file from `data/raw/announcements.parquet` on your machine. (~tens of MB; give it a minute.)

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose announcements.parquet
FNAME = list(uploaded.keys())[0]
print("uploaded:", FNAME)

## 3. Score every announcement with FinBERT
polarity = P(positive) − P(negative), per announcement. Runs on the GPU in batches with a progress counter. Expect a few minutes for ~69k rows.

In [ ]:
import pandas as pd, numpy as np
from transformers import pipeline

df = pd.read_parquet(FNAME)
df["text"] = df["text"].fillna("").astype(str)
print(f"Loaded {len(df):,} announcements")

device = 0 if torch.cuda.is_available() else -1
clf = pipeline("text-classification", model="ProsusAI/finbert",
               top_k=None, truncation=True, max_length=128, device=device)

texts = df["text"].tolist()
pol = np.zeros(len(texts))
CHUNK = 512
for i in range(0, len(texts), CHUNK):
    res = clf(texts[i:i+CHUNK], batch_size=64)
    for j, r in enumerate(res):
        s = {d["label"].lower(): d["score"] for d in r}
        pol[i+j] = s.get("positive", 0.0) - s.get("negative", 0.0)
    print(f"scored {min(i+CHUNK, len(texts)):,}/{len(texts):,}", end="\r")
print("\ndone.")

## 4. Aggregate to a daily per-stock signal
Same schema as the lexicon run, so it's a drop-in replacement.

In [ ]:
d = df.copy(); d["polarity"] = pol
d["date"] = pd.to_datetime(d["date"], errors="coerce")
d = d.dropna(subset=["date"])
g = d.groupby(["ticker", d["date"].dt.normalize()])
daily = g.agg(
    news_count   =("polarity", "size"),
    news_sentiment=("polarity", "mean"),
    news_neg_ratio=("polarity", lambda s: float((s < 0).mean())),
    news_worst   =("polarity", "min"),
    news_abs_max =("polarity", lambda s: float(np.abs(s).max())),
).reset_index()
daily.to_parquet("news_signal.parquet")

print("rows:", len(daily))
print(daily["news_sentiment"].describe()[["mean","std","min","max"]])
print("\nMost negative FinBERT news-days (should read like real bad news):")
print(daily.nsmallest(8, "news_sentiment")[["ticker","date","news_count","news_sentiment"]].to_string(index=False))

## 5. Download the result
Then put `news_signal.parquet` into your local `data/processed/` (overwrite the lexicon one).

In [ ]:
from google.colab import files
files.download("news_signal.parquet")